In [1]:
!nvidia-smi

Sat Dec 13 13:56:39 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Training

## Preparation

In [2]:
!git clone https://github.com/upiupiup/projectakhir-dl-sam-pytorch.git

Cloning into 'projectakhir-dl-sam-pytorch'...
remote: Enumerating objects: 257, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 257 (delta 93), reused 80 (delta 72), pack-reused 140 (from 3)
Receiving objects: 100% (257/257), 162.80 MiB | 37.09 MiB/s, done.
Resolving deltas: 100% (115/115), done.
Updating files: 100% (44/44), done.


In [3]:
%cd projectakhir-dl-sam-pytorch

/kaggle/working/projectakhir-dl-sam-pytorch


In [ ]:
pip install pillow

In [ ]:
import torch
import torchvision

In [ ]:
!ls

Helper Function

In [4]:
import os

PROJECT = "/kaggle/working/projectakhir-dl-sam-pytorch"
datasets = ["cifar10", "svhn"]
optimizers = ["sgd", "sam_sgd"]
seeds = [0, 1, 2]
epochs = 20

def run_once(dataset, optimizer, seed, epochs):
    out_dir = f"/kaggle/working/results/{dataset}_{optimizer}_seed{seed}"
    os.makedirs(out_dir, exist_ok=True)

    cmd = f"""
PYTHONPATH={PROJECT} \
python {PROJECT}/example/train.py \
    --dataset {dataset} \
    --model resnet18 \
    --optimizer {optimizer} \
    --epochs {epochs} \
    --batch_size 512 \
    --seed {seed} \
    --output_dir {out_dir}
    """
    print(f"\n=== RUN: dataset={dataset}, opt={optimizer}, seed={seed} ===")
    os.system(cmd)

## Sanity Check

In [ ]:
!PYTHONPATH=/kaggle/working/projectakhir-dl-sam-pytorch \
  python example/train.py \
    --dataset cifar10 \
    --model resnet18 \
    --optimizer sgd \
    --epochs 10 \
    --batch_size 128 \
    --output_dir /kaggle/working/results

## CIFAR10

### SGD

In [ ]:
for opt in ["sgd", "sam_sgd"]:
    run_once("cifar10", opt, seed, epochs=30)

In [ ]:
%cd /kaggle/working
!zip -r results_30epoch.zip results/

### AdamW

In [ ]:
for opt in ["adamw", "sam_adamw"]:
    run_once("cifar10", opt, seed, epochs=20)

In [ ]:
%cd /kaggle/working
!zip -r adamw_results.zip results/cifar10_adamw_seed0 results/cifar10_sam_adamw_seed0

### Seed

In [ ]:
for seed in [0, 1, 2]:
    for opt in ["sgd", "sam_sgd"]:
        run_once("cifar10", opt, seed, epochs=30)

In [ ]:
%cd /kaggle/working
!zip -r results_3seed_cifar10_sgd_samsgd.zip results/

In [5]:
for seed in [0]:
    for opt in ["adamw", "sam_adamw"]:
        run_once("cifar10", opt, seed, epochs=30)


=== RUN: dataset=cifar10, opt=adamw, seed=0 ===


100%|██████████| 170M/170M [00:02<00:00, 85.0MB/s] 


┏━━━━━━━━━━━━━━┳━━━━━━━╸T╺╸R╺╸A╺╸I╺╸N╺━━━━━━━┳━━━━━━━╸S╺╸T╺╸A╺╸T╺╸S╺━━━━━━━┳━━━━━━━╸V╺╸A╺╸L╺╸I╺╸D╺━━━━━━━┓
┃              ┃              ╷              ┃              ╷              ┃              ╷              ┃
┃       epoch  ┃        loss  │    accuracy  ┃        l.r.  │     elapsed  ┃        loss  │    accuracy  ┃
┠──────────────╂──────────────┼──────────────╂──────────────┼──────────────╂──────────────┼──────────────┨
┃           0  ┃      2.4526  │     14.89 %  ┃   1.000e-01  │   01:16 min  ┃┈████████████████████████▒┈┈┈┨      1.7053  │     19.47 %  ┃
┃           1  ┃      1.5231  │     22.84 %  ┃   1.000e-01  │   00:58 min  ┃┈████████████████████████▒┈┈┈┨      1.4660  │     27.48 %  ┃
┃           2  ┃      1.3845  │     31.36 %  ┃   1.000e-01  │   00:58 min  ┃┈████████████████████████▒┈┈┈┨      1.2943  │     37.62 %  ┃
┃           3  ┃      1.2622  │     38.16 %  ┃   1.000e-01  │   00:58 min  ┃┈████████████████████████▒┈┈┈┨      1.1476  │     44.31 %  ┃
┃           4  ┃      1.

In [8]:
%cd /kaggle/working/results

!zip -r cifar10_adamw_vs_sam_adamw_seed0.zip \
    cifar10_adamw_seed0 \
    cifar10_sam_adamw_seed0

/kaggle/working/results
  adding: cifar10_adamw_seed0/ (stored 0%)
  adding: cifar10_adamw_seed0/summary.csv (deflated 36%)
  adding: cifar10_adamw_seed0/checkpoints/ (stored 0%)
  adding: cifar10_adamw_seed0/checkpoints/cifar10_resnet18_adamw_lr0.1_wd0.0005_bs512_seed0_best.pt (deflated 25%)
  adding: cifar10_adamw_seed0/epochs/ (stored 0%)
  adding: cifar10_adamw_seed0/epochs/cifar10_resnet18_adamw_lr0.1_wd0.0005_bs512_seed0_20251213-135653.csv (deflated 60%)
  adding: cifar10_sam_adamw_seed0/ (stored 0%)
  adding: cifar10_sam_adamw_seed0/summary.csv (deflated 38%)
  adding: cifar10_sam_adamw_seed0/checkpoints/ (stored 0%)
  adding: cifar10_sam_adamw_seed0/checkpoints/cifar10_resnet18_sam_adamw_rho0.05_adaptive0_lr0.1_wd0.0005_bs512_seed0_best.pt (deflated 9%)
  adding: cifar10_sam_adamw_seed0/epochs/ (stored 0%)
  adding: cifar10_sam_adamw_seed0/epochs/cifar10_resnet18_sam_adamw_rho0.05_adaptive0_lr0.1_wd0.0005_bs512_seed0_20251213-142914.csv (deflated 60%)


In [9]:
for seed in [1, 2]:
    for opt in ["adamw"]:
        run_once("cifar10", opt, seed, epochs=30)


=== RUN: dataset=cifar10, opt=adamw, seed=1 ===


100%|██████████| 170M/170M [00:01<00:00, 104MB/s]  


┏━━━━━━━━━━━━━━┳━━━━━━━╸T╺╸R╺╸A╺╸I╺╸N╺━━━━━━━┳━━━━━━━╸S╺╸T╺╸A╺╸T╺╸S╺━━━━━━━┳━━━━━━━╸V╺╸A╺╸L╺╸I╺╸D╺━━━━━━━┓
┃              ┃              ╷              ┃              ╷              ┃              ╷              ┃
┃       epoch  ┃        loss  │    accuracy  ┃        l.r.  │     elapsed  ┃        loss  │    accuracy  ┃
┠──────────────╂──────────────┼──────────────╂──────────────┼──────────────╂──────────────┼──────────────┨
┃           0  ┃      2.6217  │     12.50 %  ┃   1.000e-01  │   01:12 min  ┃┈████████████████████████▒┈┈┈┨      1.6125  │     19.38 %  ┃
┃           1  ┃      1.5384  │     22.32 %  ┃   1.000e-01  │   00:57 min  ┃┈████████████████████████▒┈┈┈┨      1.4887  │     26.27 %  ┃
┃           2  ┃      1.4292  │     30.02 %  ┃   1.000e-01  │   00:58 min  ┃┈████████████████████████▒┈┈┈┨      1.4267  │     33.84 %  ┃
┃           3  ┃      1.3416  │     34.37 %  ┃   1.000e-01  │   00:58 min  ┃┈████████████████████████▒┈┈┈┨      1.3271  │     36.67 %  ┃
┃           4  ┃      1.

In [10]:
%cd /kaggle/working/results

!zip -r cifar10_adamw.zip \
    cifar10_adamw_seed1 \
    cifar10_adamw_seed2

/kaggle/working/results
  adding: cifar10_adamw_seed1/ (stored 0%)
  adding: cifar10_adamw_seed1/summary.csv (deflated 37%)
  adding: cifar10_adamw_seed1/checkpoints/ (stored 0%)
  adding: cifar10_adamw_seed1/checkpoints/cifar10_resnet18_adamw_lr0.1_wd0.0005_bs512_seed1_best.pt (deflated 25%)
  adding: cifar10_adamw_seed1/epochs/ (stored 0%)
  adding: cifar10_adamw_seed1/epochs/cifar10_resnet18_adamw_lr0.1_wd0.0005_bs512_seed1_20251213-153519.csv (deflated 59%)
  adding: cifar10_adamw_seed2/ (stored 0%)
  adding: cifar10_adamw_seed2/summary.csv (deflated 38%)
  adding: cifar10_adamw_seed2/checkpoints/ (stored 0%)
  adding: cifar10_adamw_seed2/checkpoints/cifar10_resnet18_adamw_lr0.1_wd0.0005_bs512_seed2_best.pt (deflated 25%)
  adding: cifar10_adamw_seed2/epochs/ (stored 0%)
  adding: cifar10_adamw_seed2/epochs/cifar10_resnet18_adamw_lr0.1_wd0.0005_bs512_seed2_20251213-160725.csv (deflated 60%)


In [ ]:
%cd /kaggle/working/results

!zip -r cifar10_sam_adamw_seed1_2.zip \
    cifar10_sam_adamw_seed1 \
    cifar10_sam_adamw_seed2

## SVHN

### SGD

In [ ]:
for opt in ["sgd", "sam_sgd"]:
    run_once("svhn", opt, seed, epochs=20)

In [ ]:
%cd /kaggle/working/projectakhir-dl-sam-pytorch

import sys

ROOT = "/kaggle/working/projectakhir-dl-sam-pytorch"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from example.data.svhn import Svhn

# bikin loader SVHN
svhn = Svhn(batch_size=128, threads=0)

train_loader = svhn.train
test_loader  = svhn.test

print("Jumlah sample TRAIN :", len(train_loader.dataset))
print("Jumlah sample TEST  :", len(test_loader.dataset))

# cek contoh batch
images, labels = next(iter(train_loader))
print("Batch image shape :", images.shape)   # [B, 3, 32, 32]
print("Batch label shape :", labels.shape)

### AdamW